In [ ]:
import os
import base64
import mimetypes
from pydantic import BaseModel, Field
from langchain_google_vertexai import ChatVertexAI
from langchain_core.messages import HumanMessage
from finance_analysis.config import global_config as glob

def file_to_gemini_part(file_path: str) -> dict:
    mime_type = mimetypes.guess_type(file_path)[0] or "application/octet-stream"
    with open(file_path, "rb") as input_file:
        file_bytes = input_file.read()

    if mime_type.startswith("image/"):
        encoded = base64.b64encode(file_bytes).decode("utf-8")
        return {
            "type": "image_url",
            "image_url": {
                "url": f"data:{mime_type};base64,{encoded}",
            },
        }

    return {
        "type": "media",
        "mime_type": mime_type,
        "data": file_bytes,
    }

class InvoiceExtraction(BaseModel):
    vendor_name: str | None = Field(default=None, description="Vendor or company name on the invoice")
    invoice_number: str | None = Field(default=None, description="Invoice number or reference number")
    invoice_date: str | None = Field(default=None, description="Invoice issue date in ISO format when possible, for example 2026-03-19")
    currency: str | None = Field(default=None, description="Invoice currency code such as EUR, USD, or HUF")
    total_amount: float | None = Field(default=None, description="Final total amount on the invoice as a number")
    guest_name: str | None = Field(default=None, description="Guest or traveler name if present")
    arrival_date: str | None = Field(default=None, description="Arrival or check-in date in ISO format when possible")
    departure_date: str | None = Field(default=None, description="Departure or check-out date in ISO format when possible")

invoice_name = "Hotel_Invoice.pdf"
invoice_name = "Rechnung_Hotel.pdf"  


file_path = os.path.join(glob.DATA_PKG_DIR, invoice_name)

llm = ChatVertexAI(
    project=glob.GCP_PROJECT,
    model_name="gemini-2.5-flash",
    temperature=0.0,
    max_retries=2,
)

structured_llm = llm.with_structured_output(InvoiceExtraction)

response = structured_llm.invoke([
    HumanMessage(
        content=[
            file_to_gemini_part(file_path),
            {
                "type": "text",
                "text": "Extract the invoice fields from this document. Leave missing fields empty.",
            },
        ]
    )
])

response.model_dump()